In [4]:
pip install rasterio netCDF4 satpy xarray opencv-python

Note: you may need to restart the kernel to use updated packages.


In [2]:
"""
applied_mask.py
===============
Módulo de aplicação de máscaras de nuvem reclassificadas sobre imagens de
radiância/reflectância do produto ABI-L2-CMIPF do satélite GOES-16.

Fluxo principal:
    1. Pareamento automático entre arquivos de máscara (.tif) e de cena (.nc)
       pelo identificador temporal do nome do arquivo GOES-16 (_sYYYYDDDHHMMSS).
    2. Leitura da máscara reclassificada: pixels de nuvem (valor 0) tornam-se NaN.
    3. Leitura da variável principal da cena NetCDF4.
    4. Redimensionamento da cena para a grade da máscara.
    5. Multiplicação elementar: pixels sob nuvem recebem NaN no resultado.
    6. Exportação como GeoTIFF georreferenciado (float32, compressão LZW).

Convenção de máscara:
    1 → pixel válido (céu claro)  → mantido no resultado
    0 → pixel inválido (nuvem)    → substituído por NaN

Dependências:
    numpy, rasterio, xarray, opencv-python

Uso típico:
    batch_process(
        mask_files=mask_files,
        scene_files=scene_files,
        output_dir='MASCARA_APLICADA',
    )
"""

import os
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import rasterio
import xarray as xr

# ---------------------------------------------------------------------------
# Constantes do módulo
# ---------------------------------------------------------------------------

# Padrão de data nos nomes de arquivo GOES-16: _sYYYYDDDHHMMSS
_PADRAO_CHAVE = re.compile(r"G16_s(\d+)_e")

# Interpolação padrão para redimensionamento de dados de radiância.
# INTER_LINEAR é aceitável para dados contínuos; use INTER_NEAREST para
# máscaras ou dados categóricos.
DEFAULT_INTERPOLATION = cv2.INTER_LINEAR

# dtype de saída: float32 é suficiente para dados de radiância GOES-16
# e reduz o tamanho do arquivo em ~50% em relação a float64.
OUTPUT_DTYPE = np.float32


# ---------------------------------------------------------------------------
# Funções auxiliares (uso interno)
# ---------------------------------------------------------------------------

def _extract_key(filename: str) -> Optional[str]:
    """
    Extrai o identificador temporal do nome de um arquivo GOES-16.

    O padrão esperado é `G16_s<CHAVE>_e`, onde CHAVE é uma sequência
    numérica que identifica unicamente o instante de observação.

    Parâmetros:
        filename (str): Nome do arquivo (sem caminho completo).

    Retorna:
        str | None: Chave temporal extraída, ou None se não encontrada.

    Exemplos:
        >>> _extract_key('OR_ABI-L2-CMIPF_G16_s20201821600164_e20201821609472.nc')
        '20201821600164'
    """
    match = _PADRAO_CHAVE.search(filename)
    return match.group(1) if match else None


def _build_output_filename(scene_path: str) -> str:
    """
    Constrói o nome do arquivo GeoTIFF de saída a partir do nome da cena.

    Estratégia: remove o prefixo 'OR_' e a extensão '.nc', substituindo
    por '.tif'. Usa `parts[1:6]` para manter o padrão GOES-16 canônico
    (tipo_nível_produto_satélite_timestamp). Se o nome não seguir o padrão
    esperado (menos de 6 partes), usa o stem completo como fallback e
    emite um aviso.

    Parâmetros:
        scene_path (str): Caminho completo ou nome do arquivo de cena `.nc`.

    Retorna:
        str: Nome do arquivo de saída com extensão `.tif`.
    """
    stem = Path(scene_path).stem          # nome sem extensão
    parts = stem.split("_")

    if len(parts) >= 6:
        base_name = "_".join(parts[1:6])
    else:
        # Fora do padrão esperado: usa o nome completo sem extensão
        base_name = stem
        print(f"  ⚠️  Nome fora do padrão GOES-16, usando fallback: '{base_name}'")

    return f"{base_name}.tif"


# ---------------------------------------------------------------------------
# Funções públicas
# ---------------------------------------------------------------------------

def match_files(
    mask_files: List[str],
    scene_files: List[str],
) -> List[Tuple[str, str]]:
    """
    Emparelha arquivos de máscara e de cena pelo identificador temporal GOES-16.

    O pareamento usa a chave `G16_s<CHAVE>_e` presente em ambos os nomes.
    Arquivos sem a chave esperada são ignorados silenciosamente.

    Parâmetros:
        mask_files  (list[str]): Caminhos dos arquivos de máscara (.tif).
        scene_files (list[str]): Caminhos dos arquivos de cena (.nc).

    Retorna:
        list[tuple[str, str]]: Pares ordenados (mask_path, scene_path)
                               onde a chave temporal coincide.

    Exemplos:
        >>> pairs = match_files(mask_files, scene_files)
        >>> print(f"{len(pairs)} pares encontrados.")
    """
    # Indexa máscaras pela chave temporal
    mask_index: Dict[str, str] = {}
    for path in mask_files:
        key = _extract_key(os.path.basename(path))
        if key:
            mask_index[key] = path

    # Encontra cenas com chave correspondente
    pairs = []
    for path in scene_files:
        key = _extract_key(os.path.basename(path))
        if key and key in mask_index:
            pairs.append((mask_index[key], path))

    return pairs


def process_and_export(
    mask_path: str,
    scene_path: str,
    output_dir: str,
    interpolation: int = DEFAULT_INTERPOLATION,
) -> None:
    """
    Aplica uma máscara de nuvem sobre uma cena e exporta o resultado como GeoTIFF.

    Pipeline:
        1. Lê a máscara reclassificada: valor 0 → NaN (pixel inválido).
        2. Lê a variável principal da cena NetCDF4.
        3. Redimensiona a cena para a grade da máscara.
        4. Multiplica: pixels sob nuvem recebem NaN no resultado.
        5. Converte para float32 e exporta com georreferenciamento da máscara.

    Parâmetros:
        mask_path     (str): Caminho do arquivo de máscara reclassificada (.tif).
        scene_path    (str): Caminho do arquivo de cena NetCDF4 (.nc).
        output_dir    (str): Diretório de saída do GeoTIFF resultante.
        interpolation (int): Método de interpolação do cv2 para redimensionamento
                             (padrão: INTER_LINEAR para dados contínuos).

    Levanta:
        FileNotFoundError: Se máscara ou cena não existirem.
    """
    # Valida existência dos arquivos de entrada
    for path in (mask_path, scene_path):
        if not os.path.exists(path):
            raise FileNotFoundError(f"Arquivo de entrada não encontrado: '{path}'")

    output_filename = _build_output_filename(scene_path)
    output_path     = os.path.join(output_dir, output_filename)

    # 1. Leitura da máscara: valor 0 → NaN
    with rasterio.open(mask_path) as src:
        mask_data = src.read(1).astype(OUTPUT_DTYPE)
        profile   = src.profile.copy()

    # Pixels de nuvem (valor 0) tornam-se NaN para propagar a invalidade
    mask_data = np.where(mask_data == 0, np.nan, mask_data)

    # 2. Leitura da cena — .values garante carregamento imediato em memória
    with xr.open_dataset(scene_path) as ds:
        first_var  = next(iter(ds.variables))
        scene_data = ds[first_var].values.astype(OUTPUT_DTYPE)

    # 3. Redimensionamento da cena para a grade da máscara
    target_height, target_width = mask_data.shape
    scene_data = cv2.resize(
        scene_data,
        (target_width, target_height),
        interpolation=interpolation,
    )

    # 4. Aplicação da máscara: multiplicação propaga NaN nos pixels de nuvem
    result = scene_data * mask_data

    # 5. Exportação — dtype fixado em float32 independente do dtype intermediário
    profile.update({
        "driver":   "GTiff",
        "count":    1,
        "dtype":    "float32",
        "compress": "lzw",
        "tiled":    True,
        "nodata":   np.nan,
    })

    # Garante existência do diretório de saída (protegido contra path vazio)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(result, 1)


def batch_process(
    mask_files: List[str],
    scene_files: List[str],
    output_dir: str,
    start_index: int = 0,
    interpolation: int = DEFAULT_INTERPOLATION,
    verbose: bool = True,
) -> int:
    """
    Processa em lote os pares de máscara/cena correspondentes.

    Parâmetros:
        mask_files    (list[str]): Caminhos dos arquivos de máscara (.tif).
        scene_files   (list[str]): Caminhos dos arquivos de cena (.nc).
        output_dir    (str):       Diretório de saída dos GeoTIFFs.
        start_index   (int):       Índice inicial do processamento (padrão: 0).
                                   Útil para retomar após interrupção.
        interpolation (int):       Método de interpolação cv2 (padrão: INTER_LINEAR).
        verbose       (bool):      Se True, exibe progresso por arquivo.

    Retorna:
        int: Número de pares processados com sucesso.

    Exemplos:
        >>> n = batch_process(mask_files, scene_files, 'MASCARA_APLICADA')
        >>> print(f"{n} pares processados com sucesso.")
    """
    pairs = match_files(mask_files, scene_files)

    if not pairs:
        print("⚠️  Nenhum par correspondente encontrado.")
        return 0

    # Aplica o offset de início
    pairs_to_process = pairs[start_index:]
    total_pairs      = len(pairs)
    total_to_process = len(pairs_to_process)

    if verbose:
        print(f"Pares encontrados  : {total_pairs}")
        print(f"A processar        : {total_to_process} (a partir do índice {start_index})")

    success = 0
    for i, (mask_file, scene_file) in enumerate(pairs_to_process, start=start_index + 1):
        try:
            process_and_export(mask_file, scene_file, output_dir, interpolation)
            success += 1
            if verbose:
                out_name = _build_output_filename(scene_file)
                print(f"  ✔ [{i}/{total_pairs}] {os.path.basename(scene_file)} → {out_name}")
        except Exception as e:
            print(f"  ✘ [{i}/{total_pairs}] {os.path.basename(scene_file)} — Erro: {e}")

    print(f"\nConcluído: {success}/{total_to_process} pares processados com sucesso.")
    return success


# ---------------------------------------------------------------------------
# Exemplo de uso
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    MASK_DIR   = Path("MASCARA_RECLASSIFICADA")
    SCENE_DIR  = Path("Dados") / "ABI-L2-CMIPF" / "netCDF"
    OUTPUT_DIR = Path("MASCARA_APLICADA")

    # Listagem ordenada garante comportamento determinístico entre execuções
    mask_files = sorted(
        str(f) for f in MASK_DIR.iterdir()
        if f.is_file() and f.suffix == ".tif"
    )
    scene_files = sorted(
        str(f) for f in SCENE_DIR.iterdir()
        if f.is_file() and f.suffix == ".nc"
    )

    if not mask_files or not scene_files:
        print("⚠️  Nenhum arquivo encontrado em um ou ambos os diretórios.")
    else:
        batch_process(
            mask_files=mask_files,
            scene_files=scene_files,
            output_dir=str(OUTPUT_DIR),
            start_index=0,
        )

Pares encontrados  : 60
A processar        : 60 (a partir do índice 0)
  ✔ [1/60] OR_ABI-L2-CMIPF-M6C01_G16_s20201501300166_e20201501309474_c20201501309548.nc → ABI-L2-CMIPF-M6C01_G16_s20201501300166_e20201501309474_c20201501309548.tif
  ✔ [2/60] OR_ABI-L2-CMIPF-M6C01_G16_s20201501310166_e20201501319474_c20201501319550.nc → ABI-L2-CMIPF-M6C01_G16_s20201501310166_e20201501319474_c20201501319550.tif
  ✔ [3/60] OR_ABI-L2-CMIPF-M6C01_G16_s20201501320166_e20201501329474_c20201501329547.nc → ABI-L2-CMIPF-M6C01_G16_s20201501320166_e20201501329474_c20201501329547.tif
  ✔ [4/60] OR_ABI-L2-CMIPF-M6C01_G16_s20201501330166_e20201501339474_c20201501339551.nc → ABI-L2-CMIPF-M6C01_G16_s20201501330166_e20201501339474_c20201501339551.tif
  ✔ [5/60] OR_ABI-L2-CMIPF-M6C01_G16_s20201501340166_e20201501349474_c20201501349555.nc → ABI-L2-CMIPF-M6C01_G16_s20201501340166_e20201501349474_c20201501349555.tif
  ✔ [6/60] OR_ABI-L2-CMIPF-M6C01_G16_s20201501350166_e20201501359474_c20201501359555.nc → ABI-L2-CMIPF-M